# 02 — Fine-tuned multilingual encoder (mDeBERTa-v3 / XLM-R)
**Kaggle settings:** Accelerator = GPU T4 x2, Internet = On (to download weights).
**Inputs:** the competition data + the output of notebook 01 (`folds.csv`).
**Outputs:** `val_<tag>.csv`, `test_<tag>.csv` (logits), `submission_<tag>.csv`, `long_text_results.csv`.

Training uses head+tail truncation (first HEAD and last MAX_LEN-2-HEAD tokens).
At inference we compare three ways of handling long texts on the same validation split:
head+tail, mean over sliding chunks, and the average of both.

In [1]:
import os, glob, time, math, gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import f1_score, classification_report, confusion_matrix

INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
def find(name):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    assert hits, f"{name} not found under {INPUT_ROOT}: add it via '+ Add Input'"
    return hits[0]

MODEL_NAME = "xlm-roberta-base"
TAG = "xlmr"
MAX_LEN, HEAD = 512, 256       # head+tail: 256 first + 254 last tokens
N_TRAIN = 48000                # rows of the training part used for fine-tuning (fixed seed)
EPOCHS, LR, BS, ACC = 2, 2e-5, 8, 2   # per-device batch 8 x 2 GPUs x accumulation 2 = 32
MAX_CHUNKS, STRIDE = 8, 384    # sliding-window inference for long texts
MAX_CHARS = 30000
SEED = 42
LABELS = ["human", "ai", "fusion"]; L2I = {l: i for i, l in enumerate(LABELS)}
set_seed(SEED)
print(torch.cuda.device_count(), "GPU(s)")

2 GPU(s)


In [2]:
train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
sample_sub = pd.read_csv(find("sample_submission.csv"))
folds = pd.read_csv(find("folds.csv"))[["id", "fold", "group"]]
work = folds.merge(train, on="id", how="left")
tr = work[work.fold != 0].sample(frac=1, random_state=SEED)
tr = tr.groupby("label", group_keys=False).sample(frac=min(1.0, N_TRAIN / len(tr)), random_state=SEED)
va = work[work.fold == 0].reset_index(drop=True)
print("train:", len(tr), "val:", len(va), "test:", len(test))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
def encode(texts):
    return tok([t[:MAX_CHARS] for t in texts], add_special_tokens=False, truncation=False)["input_ids"]

def head_tail(ids):
    body = MAX_LEN - 2
    if len(ids) > body:
        ids = ids[:HEAD] + ids[-(body - HEAD):]
    return [tok.cls_token_id] + ids + [tok.sep_token_id]

def chunks(ids):
    body = MAX_LEN - 2
    starts = list(range(0, max(1, len(ids) - body + STRIDE), STRIDE))[:MAX_CHUNKS] if len(ids) > body else [0]
    return [[tok.cls_token_id] + ids[s:s + body] + [tok.sep_token_id] for s in starts]

t0 = time.time()
ids_tr, ids_va, ids_te = encode(tr.text.tolist()), encode(va.text.tolist()), encode(test.text.tolist())
print(f"tokenized in {time.time()-t0:.0f}s; val share > {MAX_LEN-2} tokens:",
      np.mean([len(x) > MAX_LEN - 2 for x in ids_va]).round(3))

class DS(Dataset):
    def __init__(self, seqs, labels=None): self.seqs, self.labels = seqs, labels
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        d = {"input_ids": self.seqs[i], "attention_mask": [1] * len(self.seqs[i])}
        if self.labels is not None: d["labels"] = int(self.labels[i])
        return d

ds_tr = DS([head_tail(x) for x in ids_tr], tr.label.map(L2I).values)
ds_va = DS([head_tail(x) for x in ids_va], va.label.map(L2I).values)


train: 47853 val: 12148 test: 25000


tokenized in 95s; val share > 510 tokens: 0.642


In [3]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).float()
def metrics(p):
    return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

import inspect
n_gpu = max(1, torch.cuda.device_count())
total_steps = math.ceil(len(ds_tr) / (BS * n_gpu * ACC)) * EPOCHS
arg_dict = dict(
    output_dir="/tmp/ckpt", per_device_train_batch_size=BS, per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACC, learning_rate=LR, num_train_epochs=EPOCHS,
    warmup_steps=int(0.1 * total_steps), weight_decay=0.01, lr_scheduler_type="linear", fp16=True,
    eval_strategy="steps", eval_steps=500, save_strategy="steps", save_steps=500, save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=100, report_to="none", group_by_length=True, dataloader_num_workers=2, seed=SEED)
supported = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" not in supported and "evaluation_strategy" in supported:
    arg_dict["evaluation_strategy"] = arg_dict.pop("eval_strategy")
print("total steps:", total_steps, "| dropped:", [k for k in arg_dict if k not in supported])
args = TrainingArguments(**{k: v for k, v in arg_dict.items() if k in supported})


trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
t0 = time.time(); trainer.train(); print(f"trained in {(time.time()-t0)/60:.1f} min")
hist = pd.DataFrame([h for h in trainer.state.log_history if "eval_macro_f1" in h])[["step", "eval_loss", "eval_macro_f1"]]
print(hist.to_string(index=False)); hist.to_csv(f"{OUT}/train_history_{TAG}.csv", index=False)

total steps: 2992 | dropped: ['group_by_length']


Step,Training Loss,Validation Loss,Macro F1
500,1.481196,0.618811,0.841773
1000,1.128108,0.716459,0.814594
1500,1.011134,0.733612,0.821522
2000,0.754747,0.476470,0.884255
2500,0.707372,0.367562,0.911813
2992,0.632939,0.419932,0.895797


trained in 138.2 min
 step  eval_loss  eval_macro_f1
  500   0.618811       0.841773
 1000   0.716459       0.814594
 1500   0.733612       0.821522
 2000   0.476470       0.884255
 2500   0.367562       0.911813
 2992   0.419932       0.895797


## Inference: head+tail vs sliding chunks

In [4]:
net = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
net = net.cuda().eval()

@torch.no_grad()
def logits_for(seqs, bs=64):
    order = np.argsort([len(s) for s in seqs]); out = np.zeros((len(seqs), 3), dtype=np.float32)
    for b in range(0, len(seqs), bs):
        idx = order[b:b + bs]; batch = [seqs[i] for i in idx]; L = max(map(len, batch))
        ii = torch.full((len(batch), L), tok.pad_token_id); am = torch.zeros((len(batch), L), dtype=torch.long)
        for j, s in enumerate(batch):
            ii[j, :len(s)] = torch.tensor(s); am[j, :len(s)] = 1
        with torch.autocast("cuda", dtype=torch.float16):
            out[idx] = net(input_ids=ii.cuda(), attention_mask=am.cuda()).logits.float().cpu().numpy()
    return out

def predict_all(ids_list):
    ht = logits_for([head_tail(x) for x in ids_list])
    ch_seqs, owner = [], []
    for i, x in enumerate(ids_list):
        for c in chunks(x): ch_seqs.append(c); owner.append(i)
    ch = logits_for(ch_seqs); owner = np.array(owner)
    ch = ch - np.log(np.exp(ch - ch.max(1, keepdims=True)).sum(1, keepdims=True)) - ch.max(1, keepdims=True)  # log-probs
    ch_mean = np.zeros_like(ht); np.add.at(ch_mean, owner, ch); ch_mean /= np.bincount(owner, minlength=len(ids_list))[:, None]
    # fusion = only part of the text rewritten -> one "fusion-looking" chunk is enough: max over chunks for that class
    fus_max = np.full(len(ids_list), -np.inf, dtype=np.float32); np.maximum.at(fus_max, owner, ch[:, 2])
    ch_maxf = ch_mean.copy(); ch_maxf[:, 2] = fus_max
    return {"head_tail": ht, "chunk_mean": ch_mean, "avg_both": (ht + ch_mean) / 2, "chunk_mean_maxfusion": ch_maxf}

t0 = time.time(); pv = predict_all(ids_va); print(f"val inference {time.time()-t0:.0f}s")
y_va = va.label.map(L2I).values
long_mask = np.array([len(x) > MAX_LEN - 2 for x in ids_va])
rows = []
for k, v in pv.items():
    v = v - v.max(1, keepdims=True); v = v - np.log(np.exp(v).sum(1, keepdims=True)); pv[k] = v   # all as log-probs
    p = v.argmax(1)
    rows.append({"method": k, "val_macro_f1": f1_score(y_va, p, average="macro"),
                 "f1_long_texts": f1_score(y_va[long_mask], p[long_mask], average="macro"),
                 "f1_short_texts": f1_score(y_va[~long_mask], p[~long_mask], average="macro")})
lt = pd.DataFrame(rows).round(4); print(lt.to_string(index=False)); lt.to_csv(f"{OUT}/long_text_results_{TAG}.csv", index=False)
best_m = lt.sort_values("val_macro_f1").iloc[-1].method
print("best long-text method:", best_m)
p = pv[best_m].argmax(1)
print(classification_report(y_va, p, target_names=LABELS, digits=4))
print(confusion_matrix(y_va, p, normalize="true").round(3))
print(va.assign(y=y_va, p=p).groupby("domain")[["y", "p"]].apply(lambda d: f1_score(d.y, d.p, average="macro")).round(4).to_string())

val inference 290s
              method  val_macro_f1  f1_long_texts  f1_short_texts
           head_tail        0.9117         0.9127           0.909
          chunk_mean        0.8531         0.8116           0.909
            avg_both        0.9169         0.9212           0.909
chunk_mean_maxfusion        0.7958         0.6999           0.909
best long-text method: avg_both
              precision    recall  f1-score   support

       human     0.9007    0.8270    0.8623      2995
          ai     0.9927    0.9938    0.9932      5456
      fusion     0.8679    0.9240    0.8951      3697

    accuracy                         0.9314     12148
   macro avg     0.9204    0.9149    0.9169     12148
weighted avg     0.9320    0.9314    0.9311     12148

[[0.827 0.006 0.167]
 [0.003 0.994 0.003]
 [0.07  0.006 0.924]]
domain
arxiv_papers              0.6749
blog_authorship_corpus    0.8504
bundestag                 0.9216
cnn_news                  0.9544
euro_court_cases          0.8738
gu

In [5]:
t0 = time.time(); pt = predict_all(ids_te); print(f"test inference {time.time()-t0:.0f}s")
cols = [f"s_{l}" for l in LABELS]
pd.DataFrame(pv[best_m], columns=cols).assign(id=va.id.values).to_csv(f"{OUT}/val_{TAG}.csv", index=False)
pd.DataFrame(pt[best_m], columns=cols).assign(id=test.id.values).to_csv(f"{OUT}/test_{TAG}.csv", index=False)
sub = sample_sub[["id"]].merge(pd.DataFrame({"id": test.id, "label": np.array(LABELS)[pt[best_m].argmax(1)]}), on="id")
assert len(sub) == len(sample_sub) and sub.label.isin(LABELS).all()
sub.to_csv(f"{OUT}/submission_{TAG}.csv", index=False)
print(sub.label.value_counts(normalize=True).round(3).to_dict())

test inference 576s
{'ai': 0.513, 'fusion': 0.258, 'human': 0.229}
